# Pretrained width compression with KD + QAT

This notebook trains a width-0.75 MobileNetV2 student using the selected FP32 model as both teacher and pretrained initializer, then applies the established mixed W4/W6/W8 + A6 QAT policy. Since torchvision has no official width-0.75 MobileNetV2 weights, compatible tensors are sliced from the ImageNet-pretrained, CIFAR-10-fine-tuned teacher. The FP32 stage uses the baseline's 60-epoch schedule, learning rate, label smoothing, and strong augmentation.

In [ ]:
import os
from kaggle_secrets import UserSecretsClient
token = UserSecretsClient().get_secret('GITHUB_TOKEN')
username, repo_name = 'AdiGiriIIT', 'CS6886--Assignment-2'
!git clone https://{token}@github.com/{username}/{repo_name}.git assignment-2
%cd assignment-2
from pathlib import Path
BASELINE_SOURCE = '/kaggle/input/datasets/adityagirishep23b048/improved-baseline/baseline.pt'
DATA_DIR = '/kaggle/input/datasets/adityagirishep23b048/cifar-data'
!python -m pip install -q PyYAML matplotlib
!mkdir -p results/checkpoints results/logs experiments/student_distillation
!cp $BASELINE_SOURCE results/checkpoints/baseline.pt
!sha256sum results/checkpoints/baseline.pt
cifar_dir = Path(DATA_DIR) / 'cifar-10-batches-py'
required = ['data_batch_1', 'data_batch_2', 'data_batch_3', 'data_batch_4', 'data_batch_5', 'test_batch', 'batches.meta']
assert all((cifar_dir / name).is_file() for name in required)

In [ ]:
!set -o pipefail; python -m unittest discover -s tests -v 2>&1 | tee results/logs/kd-correctness.log
!set -o pipefail; nvidia-smi 2>&1 | tee results/logs/kd-gpu.log

In [ ]:
WIDTH_MULT = 0.75
FP32_RUN = 'mobilenetv2-075-pretrained-kd-seed6886'
QAT_RUN = 'mobilenetv2-075-pretrained-kd-mixed-qat-seed6886'
TEACHER = 'results/checkpoints/baseline.pt'

## Stage 1: pretrained width-0.75 student

The default `teacher-slice` initialization transfers every shape-compatible prefix from the trained teacher. KD uses T=2 and alpha=0.25 so ground-truth supervision remains dominant.

In [ ]:
!set -o pipefail; python -m src.distill --stage fp32 --teacher-checkpoint {TEACHER} --data-dir "{DATA_DIR}" --width-mult {WIDTH_MULT} --student-init teacher-slice --epochs 60 --learning-rate 0.01 --alpha 0.25 --temperature 2 --label-smoothing 0.1 --freeze-backbone-epochs 2 --device cuda --run-name {FP32_RUN} 2>&1 | tee results/logs/{FP32_RUN}.log

In [ ]:
import json
fp32_dir = Path('experiments/student_distillation') / FP32_RUN
fp32_metrics = json.loads((fp32_dir / 'metrics.json').read_text())
print(json.dumps(fp32_metrics, indent=2))
init = fp32_metrics['initialization']
assert init and init['copied_values'] / init['student_values'] > 0.95, init
FP32_STUDENT = str(fp32_dir / 'best.pt')

## Stage 2: mixed-precision QAT + KD

The student follows the same 8→6→target transition and W4 pointwise / W6 depthwise / W8 stem-classifier / A6 policy used by the current compact model. Only target-precision epochs are eligible for checkpoint selection.

In [ ]:
!set -o pipefail; python -m src.distill --stage qat --teacher-checkpoint {TEACHER} --student-checkpoint {FP32_STUDENT} --data-dir "{DATA_DIR}" --width-mult {WIDTH_MULT} --epochs 12 --learning-rate 0.0003 --weight-bits 4 --activation-bits 6 --depthwise-weight-bits 6 --first-last-weight-bits 8 --transition-epochs 1 --freeze-bn-epoch 10 --alpha 0.25 --temperature 2 --device cuda --run-name {QAT_RUN} 2>&1 | tee results/logs/{QAT_RUN}.log

In [ ]:
qat_dir = Path('experiments/student_distillation') / QAT_RUN
metrics = json.loads((qat_dir / 'metrics.json').read_text())
print(json.dumps(metrics, indent=2))
assert Path(metrics['artifact']).stat().st_size == metrics['total_bytes']
print(f"Validation={metrics['best_validation_accuracy']:.2f}% size={metrics['total_bytes']/2**20:.3f} MiB weight ratio={metrics['weight_compression_ratio']:.3f}x")
WINNER_CHECKPOINT = metrics['checkpoint']

## Final held-out evaluation

Evaluate the test set once after validation has selected the target-precision checkpoint.

In [ ]:
!set -o pipefail; python -m src.evaluate --checkpoint {WINNER_CHECKPOINT} --data-dir "{DATA_DIR}" --device cuda 2>&1 | tee results/logs/{QAT_RUN}-held-out-test.log

In [ ]:
import hashlib, tarfile
from IPython.display import FileLink
paths = [fp32_dir, qat_dir, Path('results/logs/kd-correctness.log'), Path('results/logs/kd-gpu.log'), Path('results/logs') / f'{FP32_RUN}.log', Path('results/logs') / f'{QAT_RUN}.log', Path('results/logs') / f'{QAT_RUN}-held-out-test.log']
paths = [path for path in paths if path.exists()]
for path in paths:
    if path.is_file(): print(f'{hashlib.sha256(path.read_bytes()).hexdigest()}  {path}')
archive = Path(f'{QAT_RUN}-artifacts.tgz')
with tarfile.open(archive, 'w:gz') as tar:
    for path in paths: tar.add(path, arcname=str(path))
print(f'Created {archive} ({archive.stat().st_size:,} bytes)')
FileLink(str(archive))